## 04 — LSTM Model (From Scratch)

### 1. Imports & Device Setup

In [1]:
import random
import pandas as pd
import numpy as np
import pickle
import wandb
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [3]:
# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version : 2.10.0+cu128
Device          : cuda
CUDA available  : True
GPU: Tesla T4


### 2. Hyperparameters

All tuning knobs in one place. Modify these to experiment with different configurations.

In [5]:
CFG = {
    # Vocabulary
    'vocab_size'    : 10000,   # how many unique words to keep
    'max_len'       : 256,     # max tokens per sample

    # Model architecture
    'embed_dim'     : 128,     # word embedding dimensions
    'hidden_dim'    : 256,     # LSTM hidden state size
    'num_layers'    : 2,       # number of LSTM layers
    'dropout'       : 0.4,     # dropout between LSTM layers
    'num_classes'   : 5,       # A, B, C, D, E

    # Training
    'batch_size'    : 32,
    'learning_rate' : 1e-3,
    'epochs'        : 50,
    'patience'      : 7,       # early stopping patience
    'weight_decay'  : 1e-4,    # L2 regularization
    'random_state'  : 42,

    # Paths
    'data_dir'   : '/kaggle/input/competitions/smart-mcq-solver-challenge',
    'output_dir' : '/kaggle/working/outputs',
}

In [6]:
torch.manual_seed(CFG['random_state'])
np.random.seed(CFG['random_state'])

OUTPUT_DIR = Path(CFG['output_dir'])
DATA_DIR   = Path(CFG['data_dir'])

for k, v in CFG.items():
    print(f'{k:<16}: {v}')

vocab_size      : 10000
max_len         : 256
embed_dim       : 128
hidden_dim      : 256
num_layers      : 2
dropout         : 0.4
num_classes     : 5
batch_size      : 32
learning_rate   : 0.001
epochs          : 50
patience        : 7
weight_decay    : 0.0001
random_state    : 42
data_dir        : /kaggle/input/competitions/smart-mcq-solver-challenge
output_dir      : /kaggle/working/outputs


### 3. Data Loading & Stratified Split

Load raw CSVs, lowercase all text fields, then do a manual stratified 80/20 split so every answer class is proportionally represented.

In [7]:
raw_train = pd.read_csv(DATA_DIR / 'train.csv')
raw_test  = pd.read_csv(DATA_DIR / 'test.csv')

In [8]:
# Lowercase all the text
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_cols:
    raw_train[col] = raw_train[col].str.lower().str.strip()
    raw_test[col]  = raw_test[col].str.lower().str.strip()

In [9]:
# Stratified 80/20 split
np.random.seed(CFG['random_state'])
train_idx, val_idx = [], []
for ans in 'ABCDE':
    idx = raw_train[raw_train['answer'] == ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_idx += idx[:cut]
    val_idx   += idx[cut:]

train_df = raw_train.loc[train_idx].reset_index(drop=True)
val_df   = raw_train.loc[val_idx].reset_index(drop=True)
test_df  = raw_test.copy()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 1599 | Val: 401 | Test: 500


In [10]:
# Answer label maps
ANSWER_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REVERSE_MAP = {v: k for k, v in ANSWER_MAP.items()}

In [11]:
dup_mask = train_df.duplicated(subset=['prompt','A','B','C','D','E'], keep=False)
print(f"Rows involved in duplicates: {dup_mask.sum()}")
print(f"Unique duplicate groups: {train_df[dup_mask].groupby('prompt').ngroups}")

Rows involved in duplicates: 232
Unique duplicate groups: 109


### 4. Simple word-level tokenizer built from scratch.

    Workflow:
      1. build_vocab()  — scan all training texts, keep top-N words
      2. encode()       — text → list of integer token IDs
      3. pad_or_truncate() — make every sequence the same length

    Special tokens:
      <PAD> = 0  (padding for shorter sequences)
      <UNK> = 1  (unknown words not in vocab)

In [12]:
class MCQTokenizer:

    PAD_TOKEN = '<PAD>'
    UNK_TOKEN = '<UNK>'

    def __init__(self, vocab_size: int = 10000):
        self.vocab_size = vocab_size
        self.word2idx   = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.idx2word   = {0: self.PAD_TOKEN, 1: self.UNK_TOKEN}
        self.vocab_built = False

    def tokenize(self, text: str) -> List[str]:
        """Split text into lowercase words"""
        return str(text).lower().split()

    def build_vocab(self, texts: List[str]):
        """
        Build vocabulary from a list of training texts.
        Keeps only top vocab_size most frequent words.
        """
        counter = Counter()
        for text in texts:
            counter.update(self.tokenize(text))

        # Keep top (vocab_size - 2) words (reserve 0, 1 for PAD, UNK)
        most_common = counter.most_common(self.vocab_size - 2)
        for idx, (word, _) in enumerate(most_common, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx]  = word

        self.vocab_built = True
        print(f'  Vocab built: {len(self.word2idx)} tokens')

    def encode(self, text: str) -> List[int]:
        """Convert text to list of integer IDs"""
        words = self.tokenize(text)
        return [self.word2idx.get(w, 1) for w in words]  # 1 = <UNK>

    def pad_or_truncate(self, ids: List[int], max_len: int) -> List[int]:
        """Make sequence exactly max_len long (truncate or pad with 0)"""
        if len(ids) >= max_len:
            return ids[:max_len]
        return ids + [0] * (max_len - len(ids))  # 0 = <PAD>


# Build tokenizer on training data
print('Building tokenizer...')
tokenizer = MCQTokenizer(vocab_size=CFG['vocab_size'])

# Collect all training texts (prompt + options)
all_train_texts = []
for _, row in train_df.iterrows():
    all_train_texts.append(
        f"{row['prompt']} {row['A']} {row['B']} {row['C']} {row['D']} {row['E']}"
    )

tokenizer.build_vocab(all_train_texts)
print(f'Tokenizer ready! Vocab size = {len(tokenizer.word2idx)}')

Building tokenizer...
  Vocab built: 3815 tokens
Tokenizer ready! Vocab size = 3815


### 5. PYTORCH DATASET

    Custom PyTorch Dataset for MCQ questions.

    Each sample:
      - combines prompt + all 5 options into one long text
      - tokenizes it with our custom tokenizer
      - pads/truncates to max_len
      - returns (input_ids tensor, label tensor)

In [13]:
class MCQDataset(Dataset):
    def __init__(self ,df ,tokenizer ,max_len ,is_test: bool = False):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Build combined text: prompt + all options
        text = (f"{row['prompt']} "
                f"{row['A']} {row['B']} {row['C']} {row['D']} {row['E']}")

        # Tokenize → encode → pad/truncate
        ids     = self.tokenizer.encode(text)
        ids     = self.tokenizer.pad_or_truncate(ids, self.max_len)
        id_tensor = torch.tensor(ids, dtype=torch.long)

        if self.is_test:
            return id_tensor   # no label for test

        # Convert answer letter to int label
        label = ANSWER_MAP[row['answer']]
        return id_tensor, torch.tensor(label, dtype=torch.long)

In [14]:
# Create datasets
train_dataset = MCQDataset(train_df, tokenizer, CFG['max_len'], is_test=False)
val_dataset   = MCQDataset(val_df,   tokenizer, CFG['max_len'], is_test=False)
test_dataset  = MCQDataset(test_df,  tokenizer, CFG['max_len'], is_test=True)

In [15]:
# Create DataLoaders 
train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=True)

In [16]:
print(f'Datasets and DataLoaders ready!')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

Datasets and DataLoaders ready!
Train batches : 50
Val batches   : 13
Test batches  : 16


In [ ]:
# Quick sanity check on one batch
sample_ids, sample_labels = next(iter(train_loader))
print(f'Sample batch  : ids={sample_ids.shape}, labels={sample_labels.shape}')

Sample batch  : ids=torch.Size([32, 256]), labels=torch.Size([32])


### 6.  LSTM MODEL (FROM SCRATCH)

    2-layer Bidirectional LSTM for MCQ answer ranking.

    Architecture:
      [Input IDs]  →  Embedding  →  LSTM x2  →  Dropout  →  Linear  →  5 logits

    Why Bidirectional?
      - Forward LSTM reads left→right (question context)
      - Backward LSTM reads right→left (option context)
      - Concatenating both gives best results

In [ ]:
class LSTMCellScratch(nn.Module):
    """One LSTM cell for one timestep. Plain math, nothing fancy."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # 4 gates stacked together: input, forget, candidate, output
        self.weight_ih = nn.Parameter(torch.randn(4 * hidden_size, input_size) * 0.1)
        self.weight_hh = nn.Parameter(torch.randn(4 * hidden_size, hidden_size) * 0.1)
        self.bias_ih = nn.Parameter(torch.zeros(4 * hidden_size))
        self.bias_hh = nn.Parameter(torch.zeros(4 * hidden_size))

    def forward(self, x_t, h_prev, c_prev):
        gates = x_t @ self.weight_ih.T + self.bias_ih + h_prev @ self.weight_hh.T + self.bias_hh
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)
        return h_t, c_t

In [ ]:
class LSTMScratch(nn.Module):
    """Bidirectional LSTM built from scratch.
    output : (batch, seq_len, hidden_size * 2)
    hidden : (num_layers * 2, batch, hidden_size), same layout as nn.LSTM
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.0):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None

        self.forward_cells = nn.ModuleList()
        self.backward_cells = nn.ModuleList()
        for layer in range(num_layers):
            in_size = input_size if layer == 0 else hidden_size * 2
            self.forward_cells.append(LSTMCellScratch(in_size, hidden_size))
            self.backward_cells.append(LSTMCellScratch(in_size, hidden_size))

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        layer_input = x
        all_hidden = []

        for layer in range(self.num_layers):
            fwd_cell = self.forward_cells[layer]
            bwd_cell = self.backward_cells[layer]

            # forward direction, left to right
            h_f = torch.zeros(batch_size, self.hidden_size, device=x.device)
            c_f = torch.zeros_like(h_f)
            forward_outputs = []
            for t in range(seq_len):
                h_f, c_f = fwd_cell(layer_input[:, t, :], h_f, c_f)
                forward_outputs.append(h_f)

            # backward direction, right to left
            h_b = torch.zeros(batch_size, self.hidden_size, device=x.device)
            c_b = torch.zeros_like(h_b)
            backward_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_b, c_b = bwd_cell(layer_input[:, t, :], h_b, c_b)
                backward_outputs[t] = h_b

            forward_outputs = torch.stack(forward_outputs, dim=1)
            backward_outputs = torch.stack(backward_outputs, dim=1)
            layer_input = torch.cat([forward_outputs, backward_outputs], dim=2)

            all_hidden.append(h_f)
            all_hidden.append(h_b)

            if self.dropout is not None and layer < self.num_layers - 1:
                layer_input = self.dropout(layer_input)

        hidden = torch.stack(all_hidden, dim=0)
        return layer_input, hidden

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = LSTMScratch(embed_dim, hidden_dim, num_layers, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        output, hidden = self.lstm(embedded)
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        combined = torch.cat([forward_hidden, backward_hidden], dim=1)

        out = self.dropout(combined)
        logits = self.fc(out)
        return logits

In [ ]:
# class LSTMClassifier(nn.Module):
#     def __init__(self, vocab_size , embed_dim , hidden_dim , num_layers , num_classes , dropout , pad_idx:int = 0 ):
#         super().__init__()
        
#         # Layer 1: Embedding
#         # Maps each token ID → dense embed_dim-dimensional vector
#         # padding_idx=0 ensures <PAD> tokens don't affect gradients
#         self.embedding = nn.Embedding(
#             num_embeddings = vocab_size,
#             embedding_dim  = embed_dim,
#             padding_idx    = pad_idx
#         )

#         # Layer 2: LSTM
#         # bidirectional=True → output is hidden_dim * 2
#         # batch_first=True   → input shape: (batch, seq_len, embed_dim)
#         self.lstm = nn.LSTM(
#             input_size    = embed_dim,
#             hidden_size   = hidden_dim,
#             num_layers    = num_layers,
#             batch_first   = True,
#             bidirectional = True,
#             dropout       = dropout if num_layers > 1 else 0.0
#         )
        
#         # Layer 3: Dropout for regularization
#         self.dropout = nn.Dropout(dropout)

#         # Layer 4: Fully-connected head
#         # hidden_dim*2 because bidirectional → concat forward + backward
#         self.fc = nn.Linear(hidden_dim * 2, num_classes)

#     def forward(self, input_ids: torch.Tensor) -> torch.Tensor:

#         # Step 1: Embed token IDs → word vectors
#         embedded = self.embedding(input_ids)
#         embedded = self.dropout(embedded)

#         # Step 2: Pass through LSTM
#         output, (hidden, cell) = self.lstm(embedded)

#         # Step 3: Pool — take last hidden states from both directions
#         forward_hidden  = hidden[-2]   # last layer, forward
#         backward_hidden = hidden[-1]   # last layer, backward
#         combined = torch.cat([forward_hidden, backward_hidden], dim=1)

#         # Step 4: Dropout + linear projection → class logits
#         out = self.dropout(combined)
#         logits = self.fc(out)          # (batch, 5)

#         return logits

#### 6. Instantiate & Inspect

In [19]:
# Instantiate model
model = LSTMClassifier(
    vocab_size  = CFG['vocab_size'],
    embed_dim   = CFG['embed_dim'],
    hidden_dim  = CFG['hidden_dim'],
    num_layers  = CFG['num_layers'],
    num_classes = CFG['num_classes'],
    dropout     = CFG['dropout'],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' LSTM model created!')
print(f'   Architecture: Embedding → BiLSTM x2 → Dropout → FC')
print(f'   Total params: {total_params:,}')
print(f'   Device      : {DEVICE}')
print(model)

 LSTM model created!
   Architecture: Embedding → BiLSTM x2 → Dropout → FC
   Total params: 3,650,053
   Device      : cuda
LSTMClassifier(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=512, out_features=5, bias=True)
)


### 7. Training Utilities

- **MAP@3 from logits**: converts softmax probabilities to top-3 predictions and computes mean average precision
- **train_one_epoch**: single epoch with mixed-precision forward pass and gradient clipping
- **evaluate_model**: no-grad evaluation loop

In [20]:
### 1.Calculate MAP@3 directly from raw logits and label tensors

def map_at_3_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    probs  = torch.softmax(logits, dim=1).cpu().numpy()
    labels = labels.cpu().numpy()
    scores = []
    for i, true in enumerate(labels):
        top3 = np.argsort(probs[i])[-3:][::-1]
        score = (1.0 / (np.where(top3 == true)[0][0] + 1)
                 if true in top3 else 0.0)
        scores.append(score)
    return float(np.mean(scores))

In [21]:
### 2. Train for one epoch, return avg loss and MAP@3

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss, all_logits, all_labels = 0.0, [], []

    for ids, labels in loader:
        ids, labels = ids.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass (faster on GPU)
        with autocast(enabled=(device.type == 'cuda')):
            logits = model(ids)
            loss   = criterion(logits, labels)

        # Backward with gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss  += loss.item()
        all_logits.append(logits.detach())
        all_labels.append(labels.detach())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    avg_loss   = total_loss / len(loader)
    train_map3 = map_at_3_from_logits(all_logits, all_labels)

    return avg_loss, train_map3

In [22]:
### 3. Evaluate on val/test, return loss and MAP@3

@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []

    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            ids, labels = batch
            ids, labels = ids.to(device), labels.to(device)
            logits = model(ids)
            total_loss += criterion(logits, labels).item()
            all_labels.append(labels)
        else:
            ids = batch.to(device)
            logits = model(ids)

        all_logits.append(logits)

    all_logits = torch.cat(all_logits)
    avg_loss   = total_loss / len(loader)

    if all_labels:
        all_labels = torch.cat(all_labels)
        val_map3   = map_at_3_from_logits(all_logits, all_labels)
    else:
        val_map3 = 0.0

    return avg_loss, val_map3, all_logits

print('Training utilities done !')

Training utilities done !


### 8. Output Directories

In [23]:
Path("/kaggle/working/outputs/models").mkdir(parents=True ,exist_ok=True)
Path("/kaggle/working/outputs/predictions").mkdir(parents=True ,exist_ok=True)

print("Folders created!")

Folders created!


### 9. Experiment Tracking (WandB)

In [24]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    WANDB_KEY = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=WANDB_KEY)
    USE_WANDB = True
    print('WandB logged in via Kaggle secret!')
except Exception as e:
    print(f'WandB secret not found ({e}). Running without WandB.')
    USE_WANDB = False

if USE_WANDB:
    run = wandb.init(
        project = '23f2003236-t22026',
        name    = 'lstm_from_scratch_final',
        config  = CFG,
        tags    = ['lstm', 'from-scratch', 'kaggle-gpu'],
        reinit  = True
    )
    print(f'   Run URL: {run.url}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB logged in via Kaggle secret!


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260703_021136-yaabom3q
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lstm_from_scratch_final
wandb: ⭐️ View project at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/yaabom3q


   Run URL: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/yaabom3q


### 10. Training Loop

Full training with Adam optimizer, cosine-style LR scheduling via `ReduceLROnPlateau`, gradient clipping, mixed precision, early stopping, and WandB logging. Best checkpoint is saved each time validation MAP@3 improves.

In [25]:
# Optimizer, loss, scheduler
optimizer = optim.Adam(
    model.parameters(), lr= CFG['learning_rate'], weight_decay = CFG['weight_decay']
)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)
scaler = GradScaler(enabled=(DEVICE.type == 'cuda'))

In [26]:
# Training loop 
best_val_map3   = 0.0
patience_counter = 0
history         = {'train_loss': [], 'val_loss': [], 'train_map3': [], 'val_map3': []}

print(f'\nStarting training for {CFG["epochs"]} epochs (early stop patience={CFG["patience"]})')
print(f'{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>8}  {'Train MAP@3':>11}  {'Val MAP@3':>9}  {'LR':>8}')
print('-' * 65)

for epoch in range(1, CFG['epochs'] + 1):

    # Train
    train_loss, train_map3 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, DEVICE)

    # Validate
    val_loss, val_map3, _ = evaluate_model(
        model, val_loader, criterion, DEVICE)

    # Scheduler step on val MAP@3
    scheduler.step(val_map3)
    current_lr = optimizer.param_groups[0]['lr']

    # Log to WandB
    wandb.log({
        'epoch'          : epoch,
        'train/loss'     : train_loss,
        'train/map_at_3' : train_map3,
        'val/loss'       : val_loss,
        'val/map_at_3'   : val_map3,
        'lr'             : current_lr,
    })

    # Track history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_map3'].append(train_map3)
    history['val_map3'].append(val_map3)

    print(f'{epoch:>6}  {train_loss:>10.4f}  {val_loss:>8.4f}  '
          f'{train_map3:>11.4f}  {val_map3:>9.4f}  {current_lr:>8.6f}')

    # Save best model
    if val_map3 > best_val_map3:
        best_val_map3 = val_map3
        patience_counter = 0
        torch.save(model.state_dict(),
                   OUTPUT_DIR / 'models' / 'lstm_best.pt')
        print(f'NEW BEST! val MAP@3 = {best_val_map3:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= CFG['patience']:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {CFG["patience"]} epochs)')
            break

print(f'\nTraining done! Best val MAP@3 = {best_val_map3:.4f}')
wandb.log({'best_val_map_at_3': best_val_map3})
wandb.finish()


Starting training for 50 epochs (early stop patience=7)
 Epoch  Train Loss  Val Loss  Train MAP@3  Val MAP@3        LR
-----------------------------------------------------------------
     1      1.5919    1.5702       0.4261     0.4568  0.001000
NEW BEST! val MAP@3 = 0.4568
     2      1.4716    1.1869       0.5486     0.6787  0.001000
NEW BEST! val MAP@3 = 0.6787
     3      1.1268    0.8929       0.7008     0.7893  0.001000
NEW BEST! val MAP@3 = 0.7893
     4      0.8296    0.5886       0.8043     0.8899  0.001000
NEW BEST! val MAP@3 = 0.8899
     5      0.5896    0.3630       0.8733     0.9364  0.001000
NEW BEST! val MAP@3 = 0.9364
     6      0.4396    0.2756       0.9112     0.9418  0.001000
NEW BEST! val MAP@3 = 0.9418
     7      0.3037    0.1426       0.9486     0.9825  0.001000
NEW BEST! val MAP@3 = 0.9825
     8      0.2264    0.2406       0.9610     0.9618  0.001000
     9      0.1979    0.0264       0.9682     0.9988  0.001000
NEW BEST! val MAP@3 = 0.9988
    10      0.1

wandb: updating run metadata


    21      0.0185    0.0012       0.9972     1.0000  0.000250

Early stopping at epoch 21 (no improvement for 7 epochs)

Training done! Best val MAP@3 = 1.0000


wandb: uploading history steps 17-21, summary, console lines 31-38
wandb: 
wandb: Run history:
wandb: best_val_map_at_3 ▁
wandb:             epoch ▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
wandb:                lr ████████████▃▃▃▃▃▁▁▁▁
wandb:        train/loss █▇▆▅▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
wandb:    train/map_at_3 ▁▂▄▆▆▇▇██████████████
wandb:          val/loss █▆▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/map_at_3 ▁▄▅▇▇▇███████████████
wandb: 
wandb: Run summary:
wandb: best_val_map_at_3 1
wandb:             epoch 21
wandb:                lr 0.00025
wandb:        train/loss 0.01851
wandb:    train/map_at_3 0.99719
wandb:          val/loss 0.00121
wandb:      val/map_at_3 1
wandb: 
wandb: 🚀 View run lstm_from_scratch_final at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/yaabom3q
wandb: ⭐️ View project at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260703_021136-yaabom3q/lo

In [27]:
Path("/kaggle/working/outputs/logs").mkdir(
    parents=True,
    exist_ok=True
)
print("logs folder created")

logs folder created


### 11. Inference & Submission

Load the best checkpoint, run inference on the test set, and build the Kaggle submission CSV with top-3 predicted answers per question.

In [28]:
# Load best checkpoint
print('Loading best LSTM checkpoint...')
model.load_state_dict(torch.load(OUTPUT_DIR / 'models' / 'lstm_best.pt',map_location=DEVICE))

model.eval()

Loading best LSTM checkpoint...


LSTMClassifier(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=512, out_features=5, bias=True)
)

In [29]:
# Inference on test set
print('Running inference on test set...')
all_probs = []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch.to(DEVICE)
        logits = model(ids)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)

all_probs = np.vstack(all_probs)   # (500, 5)

Running inference on test set...


In [30]:
# Build submission in Kaggle format
predictions = []
for i in range(len(test_df)):
    top3 = np.argsort(all_probs[i])[-3:][::-1]
    predictions.append(' '.join([REVERSE_MAP[j] for j in top3]))

submission = pd.DataFrame({
    'ID'        : test_df['id'].values,
    'Prediction': predictions
})

In [31]:
# Validate format
errors = sum(
    1 for p in predictions
    if len(p.split()) != 3 or not all(x in 'ABCDE' for x in p.split())
)
print(f'Format errors: {errors}  (must be 0 before submitting!)')

# Save
sub_path = OUTPUT_DIR / 'predictions' / 'lstm_submission_final.csv'
submission.to_csv(sub_path, index=False)
print(f'Submission saved: {sub_path}')
print(submission.head())

Format errors: 0  (must be 0 before submitting!)
Submission saved: /kaggle/working/outputs/predictions/lstm_submission_final.csv
   ID Prediction
0   1      A B C
1   2      B A D
2   3      B A D
3   4      E B D
4   5      C E A


In [32]:
# Save tokenizer (needed for inference later)
with open(OUTPUT_DIR / 'models' / 'lstm_tokenizer_4.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('Tokenizer saved!')

Tokenizer saved!
